# 01 — Data Prep
**Goal of this notebook (Phase P1):**
1. Load the Spider dataset (our fine-tuning data) and understand its structure
2. Inspect real examples: Schema + Question → SQL
3. Build the prompt format the model will be trained on
4. Build the role-filtered Home Credit schema (for inference later)

No training here — this is about understanding and shaping the data.
Runs on CPU (local Mac).

## Load Spider
Spider: ~10k examples across 200 databases, multi-table joins included —
the standard academic benchmark for text-to-SQL. We load it straight from
the Hugging Face Hub (no manual download).

In [1]:
from datasets import load_dataset

spider = load_dataset("xlangai/spider")
print(spider)

/opt/anaconda3/envs/text2sql/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating validation split: 100%|██████████| 1034/1034 [00:00<00:00, 172640.83 examples/s]

DatasetDict({
    train: Dataset({
        features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks'],
        num_rows: 7000
    })
    validation: Dataset({
        features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks'],
        num_rows: 1034
    })
})


## Inspect one raw example
Before shaping anything, look at the actual data. Key thing to check:
does one example contain everything we said a training example needs?

In [2]:
ex = spider["train"][0]
for k, v in ex.items():
    print(f"{k}: {v}\n")

db_id: department_management

query: SELECT count(*) FROM head WHERE age  >  56

question: How many heads of the departments are older than 56 ?

query_toks: ['SELECT', 'count', '(', '*', ')', 'FROM', 'head', 'WHERE', 'age', '>', '56']

query_toks_no_value: ['select', 'count', '(', '*', ')', 'from', 'head', 'where', 'age', '>', 'value']

question_toks: ['How', 'many', 'heads', 'of', 'the', 'departments', 'are', 'older', 'than', '56', '?']



## Get the schemas (tables.json)
Spider stores schemas separately — one entry per database, linked to
examples by `db_id`. We fetch tables.json from the Spider repo on HF Hub
and inspect the schema of the database our first example points to.

In [4]:
from huggingface_hub import list_repo_files

files = list_repo_files("xlangai/spider", repo_type="dataset")
for f in files:
    print(f)

.gitattributes
README.md
spider/train-00000-of-00001.parquet
spider/validation-00000-of-00001.parquet


In [5]:
from huggingface_hub import HfApi

api = HfApi()
results = api.list_datasets(search="spider", limit=30)
for d in results:
    print(d.id)

xlangai/spider
SALT-NLP/spider_VALUE
sdotmac/the_amazing_spiderman
jm0727/spider
jm0727/spider-val
karlen532/wikisql_and_spider
Protegee/Gwen_spider_verse_dataset
CM/spider
sert121/SpiderSQL
richardr1126/spider-schema
richardr1126/spider-context-instruct
karlen532/spider
karlen532/spider_w
richardr1126/spider-context-validation
Vierza/SpiderGirl
richardr1126/spider-natsql-skeleton-context-instruct
richardr1126/spider-natsql-context-validation
richardr1126/spider-natsql-context-instruct
vaishali/spider-tableQA
iamspidermanstop/spiderman
richardr1126/spider-skeleton-context-instruct
Falah/Spiders_Classification
merledu/RV-SpiderCrab
sartmis1/text2sql-wikisql-spider
sartmis1/text2sql-spider
lamini/spider_text_to_sql
lamini/bird_spider_train_text_to_sql
tmnam20/SpiderInstruct
MrezaPRZ/GPT-fine-tuning-spider
jackielii/spider-partial-exp


In [6]:
from huggingface_hub import list_repo_files

for repo in ["richardr1126/spider-schema", "CM/spider"]:
    print(f"===== {repo} =====")
    try:
        for f in list_repo_files(repo, repo_type="dataset"):
            print(" ", f)
    except Exception as e:
        print("  failed:", e)
    print()

===== richardr1126/spider-schema =====
  .gitattributes
  README.md
  spider_schema_rows_v2.json

===== CM/spider =====
  .gitattributes
  README.md
  data/test-00000-of-00001-08ab2cb8e7e1e149.parquet
  data/train-00000-of-00021-532b78b52190d5f3.parquet
  data/train-00001-of-00021-6a6079d7518f2d17.parquet
  data/train-00002-of-00021-20876528581cc482.parquet
  data/train-00003-of-00021-94550d377d9289be.parquet
  data/train-00004-of-00021-90827643dc544f61.parquet
  data/train-00005-of-00021-4100037cdd9a6abc.parquet
  data/train-00006-of-00021-aa21b8f8c2c7b1ac.parquet
  data/train-00007-of-00021-06ae2d8e68edaa67.parquet
  data/train-00008-of-00021-e3e60d3d428e3d67.parquet
  data/train-00009-of-00021-3f8b37bab977a5ea.parquet
  data/train-00010-of-00021-939745a62b507fc5.parquet
  data/train-00011-of-00021-838b547b1e579aed.parquet
  data/train-00012-of-00021-c8f14f8c9e29123e.parquet
  data/train-00013-of-00021-5181705b6798eb53.parquet
  data/train-00014-of-00021-89b47898c819939f.parquet
  da

In [7]:
import json
from huggingface_hub import hf_hub_download

schema_path = hf_hub_download(
    repo_id="richardr1126/spider-schema",
    filename="spider_schema_rows_v2.json",
    repo_type="dataset",
)

with open(schema_path) as f:
    schema_rows = json.load(f)

print(type(schema_rows), len(schema_rows))
# peek at the first entry to learn the format
first = schema_rows[0] if isinstance(schema_rows, list) else list(schema_rows.items())[0]
print(json.dumps(first, indent=2)[:1500])

<class 'list'> 166
{
  "Schema (values (type))": "perpetrator : Perpetrator_ID (number) , People_ID (number) , Date (text) , Year (number) , Location (text) , Country (text) , Killed (number) , Injured (number) | people : People_ID (number) , Name (text) , Height (number) , Weight (number) , Home Town (text)",
  "Primary Keys": "perpetrator : Perpetrator_ID | people : People_ID",
  "Foreign Keys": "perpetrator : People_ID equals people : People_ID",
  "db_id": "perpetrator"
}


## Verify schema coverage
The schema file must cover every db_id used in our train and validation
examples — if any is missing, those examples can't be built.

In [8]:
schema_by_db = {row["db_id"]: row for row in schema_rows}

train_dbs = set(spider["train"]["db_id"])
val_dbs   = set(spider["validation"]["db_id"])

print("distinct DBs — train:", len(train_dbs), "| validation:", len(val_dbs))
print("missing from schema file — train:", train_dbs - set(schema_by_db))
print("missing from schema file — validation:", val_dbs - set(schema_by_db))

# and our earlier example's schema, as a sanity look
print("\n--- department_management ---")
print(schema_by_db["department_management"]["Schema (values (type))"])

distinct DBs — train: 140 | validation: 20
missing from schema file — train: set()
missing from schema file — validation: set()

--- department_management ---
department : Department_ID (number) , Name (text) , Creation (text) , Ranking (number) , Budget_in_Billions (number) , Num_Employees (number) | head : head_ID (number) , name (text) , born_state (text) , age (number) | management : department_ID (number) , head_ID (number) , temporary_acting (text)


## Build the prompt format
One function: db_id + question → the exact text the model will see.
Layout: one table per line, with typed columns, then primary/foreign keys,
then the question. This function is THE data-shaping decision of the project —
the same function must be used at training AND inference time (any mismatch
between the two silently hurts quality).

In [9]:
def format_schema(db_id: str) -> str:
    row = schema_by_db[db_id]
    lines = []
    for table_part in row["Schema (values (type))"].split(" | "):
        table, cols = table_part.split(" : ", 1)
        lines.append(f"Table {table.strip()}: {cols.strip()}")
    lines.append(f"Primary keys: {row['Primary Keys']}")
    fk = row.get("Foreign Keys", "").strip()
    if fk:
        lines.append(f"Foreign keys: {fk}")
    return "\n".join(lines)

def build_prompt(db_id: str, question: str) -> str:
    return (
        "Given the following database schema, write a single SQLite SELECT "
        "query that answers the question. Output only the SQL.\n\n"
        f"{format_schema(db_id)}\n\n"
        f"Question: {question}\n"
        "SQL:"
    )

# test on our first example
ex = spider["train"][0]
print(build_prompt(ex["db_id"], ex["question"]))
print("\n--- target output ---")
print(ex["query"])

Given the following database schema, write a single SQLite SELECT query that answers the question. Output only the SQL.

Table department: Department_ID (number) , Name (text) , Creation (text) , Ranking (number) , Budget_in_Billions (number) , Num_Employees (number)
Table head: head_ID (number) , name (text) , born_state (text) , age (number)
Table management: department_ID (number) , head_ID (number) , temporary_acting (text)
Primary keys: department : Department_ID | head : head_ID | management : department_ID
Foreign keys: management : head_ID equals head : head_ID | management : department_ID equals department : Department_ID

Question: How many heads of the departments are older than 56 ?
SQL:

--- target output ---
SELECT count(*) FROM head WHERE age  >  56


## Apply to the full dataset
Create `prompt` and `completion` columns for all 7,000 train + 1,034
validation examples, then save as JSON — this file is what the Colab
training notebook will load. Chat template is applied later (03), since
it depends on the model's tokenizer.

In [10]:
def to_pair(ex):
    return {
        "prompt": build_prompt(ex["db_id"], ex["question"]),
        "completion": ex["query"],
        "db_id": ex["db_id"],
    }

train_pairs = spider["train"].map(to_pair, remove_columns=spider["train"].column_names)
val_pairs   = spider["validation"].map(to_pair, remove_columns=spider["validation"].column_names)

print(train_pairs)
print(val_pairs)

import os
os.makedirs("../data/processed", exist_ok=True)
train_pairs.to_json("../data/processed/spider_train.jsonl")
val_pairs.to_json("../data/processed/spider_val.jsonl")
print("saved ✅")

Map: 100%|███████████████████████████████████| 1034/1034 [00:00<00:00, 52661.17 examples/s]


Dataset({
    features: ['db_id', 'prompt', 'completion'],
    num_rows: 7000
})
Dataset({
    features: ['db_id', 'prompt', 'completion'],
    num_rows: 1034
})


Creating json from Arrow format: 100%|██████████████████████| 2/2 [00:00<00:00, 361.38ba/s]

saved ✅


In [11]:
# find a multi-table example (query contains JOIN)
for i in range(200):
    if "JOIN" in spider["train"][i]["query"].upper():
        ex = spider["train"][i]
        break

print(train_pairs[i]["prompt"])
print("\n--- completion ---")
print(train_pairs[i]["completion"])

Given the following database schema, write a single SQLite SELECT query that answers the question. Output only the SQL.

Table department: Department_ID (number) , Name (text) , Creation (text) , Ranking (number) , Budget_in_Billions (number) , Num_Employees (number)
Table head: head_ID (number) , name (text) , born_state (text) , age (number)
Table management: department_ID (number) , head_ID (number) , temporary_acting (text)
Primary keys: department : Department_ID | head : head_ID | management : department_ID
Foreign keys: management : head_ID equals head : head_ID | management : department_ID equals department : Department_ID

Question: What are the distinct creation years of the departments managed by a secretary born in state 'Alabama'?
SQL:

--- completion ---
SELECT DISTINCT T1.creation FROM department AS T1 JOIN management AS T2 ON T1.department_id  =  T2.department_id JOIN head AS T3 ON T2.head_id  =  T3.head_id WHERE T3.born_state  =  'Alabama'


## Part 2 — Home Credit schema (inference database)
This is the database our app queries at runtime. Steps:
1. Locate and connect to the existing Home Credit SQLite DB (from the credit risk project)
2. Inspect its tables & columns
3. Write a schema file WITH human descriptions (cryptic columns like DAYS_BIRTH)
4. Define the 3 roles and their column allowlists (RBAC layer 1)
5. Create the small employees/salaries demo tables

In [12]:
import glob, os

candidates = glob.glob(os.path.expanduser("~/Documents/credit_risk_pd_scorecard/**/*.db"), recursive=True) \
           + glob.glob(os.path.expanduser("~/Documents/credit_risk_pd_scorecard/**/*.sqlite"), recursive=True)
for c in candidates:
    print(c, "-", round(os.path.getsize(c)/1e6, 1), "MB")

/Users/nikhil/Documents/credit_risk_pd_scorecard/anaconda_projects/db/project_filebrowser.db - 0.0 MB
/Users/nikhil/Documents/credit_risk_pd_scorecard/data/home_credit_data/credit_risk.db - 1796.0 MB


In [13]:
import sqlite3

DB_PATH = "/Users/nikhil/Documents/credit_risk_pd_scorecard/data/home_credit_data/credit_risk.db"
con = sqlite3.connect(DB_PATH)

tables = [r[0] for r in con.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")]
print("Tables:", tables)

for t in tables:
    n_cols = len(con.execute(f"PRAGMA table_info({t})").fetchall())
    n_rows = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"{t}: {n_cols} columns, {n_rows:,} rows")

Tables: ['app_train', 'bureau', 'credit_card', 'installments', 'pos_cash', 'previous_application']
app_train: 58 columns, 307,511 rows
bureau: 17 columns, 1,716,428 rows
credit_card: 24 columns, 3,840,312 rows
installments: 12 columns, 13,605,401 rows
pos_cash: 8 columns, 10,001,358 rows
previous_application: 37 columns, 1,670,214 rows
